In [ ]:
# Finite-difference (FD) reference solver: physical parameters and scaling.
import numpy as np
import scipy.constants as con

# --- Grid Parameters ---
b = 500e-9                         # Device thickness (500 nm), fixed
L = int(globals().get("L", 500))   # Number of grid intervals (set by 01_FD_Grid_Independence)
dx_dim = b / L                     # Physical grid spacing (1 nm for L = 500)
position = np.arange(L + 1)

# --- Device Physics (Perovskite) ---
mu_1 = 1.0e-4          # Electron Mobility [m^2/(V s)]
epsilonr_1 = 50        # Relative Permittivity
T = 300                # Temperature (K)
Nc = 2.1e24            # Effective DOS [m^-3]
diffu_con = 1e-16      # Ion Diffusion Coefficient [m^2/s]

# --- Physical Constants ---
kB = con.k
q = con.e
epsilon_0 = con.epsilon_0
Vt = (kB * T) / q      # Thermal Voltage
eps_abs = epsilonr_1 * epsilon_0

# --- Injection Barriers ---
phia = float(globals().get("phia", 0.1))
phic = float(globals().get("phic", 0.1))

Vbi = 0.0     # Built-in potential

# Applied pre-bias voltage (V). Default 0.001 V matches the PINN solver and the
# validation case in the paper; 08_Vi_Sweep and 01_FD_Grid_Independence set it explicitly.
Vi = float(globals().get("Vi", 0.001))

# --- Physical Densities (Dimensioned) ---
n0_dim = Nc * np.exp(-phia / Vt)
nL_dim = Nc * np.exp(-phic / Vt)
P0_dim = float(globals().get("P0", 1.0e21))

# ---------------------------------------------------------
# UNIVERSAL SCALING
# ---------------------------------------------------------
# C0: The density required to drop 1 Vt across the thickness 'b'
C0 = (eps_abs * Vt) / (q * b**2)

# N_ref: The absolute maximum density in the entire system
# This ensures all dimensionless densities are <= 1.0
N_ref = max(C0, P0_dim, n0_dim, nL_dim)

# --- Normalized Boundary & Bulk Values ---
n0 = n0_dim / N_ref
nL = nL_dim / N_ref
I_bulk = P0_dim / N_ref

# ---------------------------------------------------------
# NUMERICAL CONSTANTS
# ---------------------------------------------------------
# Dimensionless spacing for a [0, 1] domain
dx_norm = 1.0 / L

Cpoi = (dx_norm**2) * (N_ref / C0)

# --- Material Property Vectors (Normalized) ---
ep_r = np.ones(L + 1) * 1.0  # Normalized relative permittivity
mu_n = np.ones(L + 1) * 1.0  # Normalized electron mobility
mu_c = diffu_con / (Vt * mu_1) # Ion mobility relative to the electron moility scale

# --- Initial Guesses (Normalized to N_ref) ---
Xc_vec = np.ones(L + 1) * I_bulk
n_vec  = np.ones(L - 1) * n0

print(f"--- Scaling Summary ---")
print(f"N_ref  = {N_ref:.3e} m^-3")
print(f"C0     = {C0:.3e} m^-3")
print(f"lambda_= {np.sqrt(C0/N_ref):.3e} (Dimensionless Debye)")
print(f"n0 (scaled) = {n0:.3e}")
print(f"I (scaled)  = {I_bulk:.3e}")

print(f"Grid intervals       = {L}")
print(f"Physical spacing     = {dx_dim * 1e9:.3f} nm")
print(f"Device thickness     = {b * 1e9:.3f} nm")
print(f"Applied pre-bias Vi  = {Vi:.6f} V")

In [ ]:
# Finite-difference (FD) reference solver: Scharfetter-Gummel discretisation
# with Gummel iteration for the steady-state psi-n-I system.

import numpy as np
import pandas as pd
from scipy.sparse import diags
import matplotlib.pyplot as plt
import os

# --- Simulation Settings ---
mixing_scheme = 1
findnewxc = True
w = 0.01
max_iter = 5000

# The stiff P0 = 1e24 m^-3 case (Section 3.4 of the paper) needs a smaller
# Gummel under-relaxation factor and more iterations to converge.
if P0_dim >= 1.0e24:
    w = 0.001
    max_iter = 70000
tolerance = 1e-10

print(
    f"w={w}, max_iter={max_iter}, "
    f"tol={tolerance}"
)

# --- Bernoulli Function for Scharfetter-Gummel ---
def B(x):
    x = float(x)
    if abs(x) < 1e-8:
        return 1.0 - x/2.0 + x*x/12.0
    elif x > 50:
        return x * np.exp(-x)
    elif x < -50:
        return -x
    else:
        return x / (np.exp(x) - 1.0)

# --- Main Simulation Loop ---
vapps = [Vi]

for Va in vapps:
    print(f'Simulating at Vapp = {Va:.4f} V')
    count = 0
    breakloop = False
    converged = False

    while not breakloop:

        # 1. SOLVE POISSON EQUATION
        a_p = np.zeros(L - 1)
        b_p, c_p = np.zeros(L - 2), np.zeros(L - 2)

        for i in range(1, L - 1):
            a_p[i-1] = -1.0 * (ep_r[i] + ep_r[i-1])
            b_p[i-1] = ep_r[i]
            c_p[i-1] = ep_r[i]
        a_p[L-2] = -1.0 * (ep_r[L-1] + ep_r[L-2])

        tripoi = diags([a_p, b_p, c_p], [0, -1, 1]).toarray()

        VL = (Vbi + Va) / Vt
        boundary_p = np.zeros(L - 1)
        boundary_p[0] = 0.0 * ep_r[0]  # V0 = 0
        boundary_p[-1] = VL * ep_r[L]

        bn_p = Cpoi * (n_vec - Xc_vec[1:L]) - boundary_p

        psi_inner = np.linalg.solve(tripoi, bn_p)
        psi = np.concatenate(([0.0], psi_inner, [VL]))

        # dpsi[i] is psi[i+1] - psi[i]. Length = L.
        dpsi = np.diff(psi)

        # 2. SOLVE ELECTRON CONTINUITY
        a_n = np.zeros(L - 1)
        b_n, c_n = np.zeros(L - 2), np.zeros(L - 2)

        for i in range(1, L - 1):
            a_n[i-1] = -(mu_n[i+1] * B(-dpsi[i]) + mu_n[i] * B(dpsi[i-1]))
            b_n[i-1] = mu_n[i+1] * B(-dpsi[i])
            c_n[i-1] = mu_n[i+1] * B(dpsi[i])

        a_n[L-2] = -(mu_n[L] * B(-dpsi[L-1]) + mu_n[L-1] * B(dpsi[L-2]))

        An = diags([a_n, b_n, c_n], [0, -1, 1]).toarray()

        boundary_n = np.zeros(L - 1)
        boundary_n[0] = mu_n[1] * B(-dpsi[0]) * n0
        boundary_n[-1] = mu_n[L] * B(dpsi[-1]) * nL

        try:
            nnew = np.linalg.solve(An, -boundary_n)
        except np.linalg.LinAlgError:
            print("Electron matrix is singular")
            raise

        nnew = np.maximum(nnew, 1e-30)

        # 3. SOLVE CATION CONTINUITY
        if findnewxc:
            nI = L + 1
            a_c = np.zeros(nI)
            b_c = np.zeros(nI - 1)
            c_c = np.zeros(nI - 1)
            rhs = np.zeros(nI)

            # Left boundary row: J_{1/2} = 0
            a_c[0] = mu_c * B(+dpsi[0])
            c_c[0] = -mu_c * B(-dpsi[0])
            rhs[0] = 0.0

            # Interior rows
            for i in range(1, L):
                dpsi_L = dpsi[i-1]
                dpsi_R = dpsi[i]
                AL = mu_c * B(+dpsi_L)
                CL = mu_c * B(-dpsi_L)
                AR = mu_c * B(+dpsi_R)
                CR = mu_c * B(-dpsi_R)
                b_c[i-1] = -AL
                a_c[i]   = AR + CL
                c_c[i]   = -CR
                rhs[i]   = 0.0

            # Right boundary row: J_{L-1/2} = 0
            b_c[-1] = mu_c * B(+dpsi[L-1])
            a_c[-1] = -mu_c * B(-dpsi[L-1])
            rhs[-1] = 0.0

            Ac = diags([a_c, b_c, c_c], [0, -1, 1]).toarray()

            # Mean-ion constraint
            row = nI // 2
            Ac[row, :] = 1.0 / nI
            rhs[row]   = I_bulk

            try:
                Xcnew = np.linalg.solve(Ac, rhs)
            except np.linalg.LinAlgError:
                print("Ion matrix is singular")
                raise

            Xcnew = np.maximum(Xcnew, 1e-30)

        # 4. CONVERGENCE CHECK
        eps_conv = 1e-30
        devn   = np.max(np.abs(nnew - n_vec) / np.maximum(np.abs(n_vec), eps_conv))
        devion = np.max(np.abs(Xcnew - Xc_vec) / np.maximum(np.abs(Xc_vec), eps_conv))

        if count % 100 == 0:
            print(f'Loop:{count:3d} | dev_n: {devn:.4e} | dev_ion: {devion:.4e}')

        if devn <= tolerance and devion <= tolerance:
            n_vec, Xc_vec = nnew, Xcnew
            breakloop = True
        else:
            n_vec = (n_vec * (1 - w)) + (w * nnew)
            if findnewxc:
                Xc_vec = (Xc_vec * (1 - w)) + (w * Xcnew)

        count += 1
        if count > max_iter:
            print(f"Reached max iterations ({max_iter})")
            break

    # --------- export CSVs (Using Universal N_max) ----------
    outdir = globals().get("OUTPUT_DIR", "outputs")
    os.makedirs(outdir, exist_ok=True)

    pd.DataFrame({'x_nm': np.arange(0, L+1, dtype=float) * (500.0 / L), 'psi_V': psi * Vt}).to_csv(os.path.join(outdir, "fd_psi.csv"), index=False)
    pd.DataFrame({'x_nm': np.arange(1, L, dtype=float) * (500.0 / L), 'n_m3': nnew * N_ref}).to_csv(os.path.join(outdir, "fd_n.csv"), index=False)
    pd.DataFrame({'x_nm': np.arange(0, L+1, dtype=float) * (500.0 / L), 'I_m3': Xc_vec * N_ref}).to_csv(os.path.join(outdir, "fd_I.csv"), index=False)

    # --------- plots ----------
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.plot(psi*Vt); plt.title('Potential (V)')

    plt.subplot(1, 3, 2)
    plt.semilogy(nnew*N_ref); plt.title('n (m^-3)')

    plt.subplot(1, 3, 3)
    plt.semilogy(Xc_vec*N_ref); plt.title('I (m^-3)')
    plt.tight_layout(); plt.show()